In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

import os

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

/home/kejh66/anaconda3/envs/lad/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/kejh66/anaconda3/envs/lad/lib/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

2025-11-04 11:19:53.567870: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.52it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


In [3]:
log_datasets = {
    "android": "/home/kejh66/work/MLLM-MAD/dataset/sampled_logs/Android_labeling_sample.txt",
    "apache": "/home/kejh66/work/MLLM-MAD/dataset/sampled_logs/Apache_labeling_sample.txt",
    "proxifier": "/home/kejh66/work/MLLM-MAD/dataset/sampled_logs/Proxifier_labeling_sample.txt"
}

labels_datasets = {
    "android": "/home/kejh66/work/MLLM-MAD/dataset/sampled_logs/Android_labeling_sample_labels.txt",
    "apache": "/home/kejh66/work/MLLM-MAD/dataset/sampled_logs/Apache_labeling_sample_labels.txt",
    "proxifier": "/home/kejh66/work/MLLM-MAD/dataset/sampled_logs/Proxifier_labeling_sample_labels.txt"
}

In [4]:
def read_log_file(dataset_name, dataset_dict):
    if dataset_name not in dataset_dict:
        raise ValueError(f"Dataset '{dataset_name}' not found. Available: {list(dataset_dict.keys())}")
    
    file_path = dataset_dict[dataset_name]
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File does not exist: {file_path}")
    
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.rstrip("\n") for line in f if line.strip()]
    
    return lines

In [5]:
def build_messages(log_lines):
    indexed_logs = "\n".join([f"{i+1}. {line}" for i, line in enumerate(log_lines)])
    
    prompt_text = f"""
You are an expert system log anomaly detector.

Analyze each log entry and determine whether it represents NORMAL or ABNORMAL system behavior.

Rules:
• NORMAL: Regular operational events such as successful startups, connections, completions, or expected state updates.
• ABNORMAL: Any indication of errors, failures, crashes, exceptions, timeouts, unexpected shutdowns, or suspicious activities.
• If a line is ABNORMAL, provide a brief but informative explanation describing the reason for the anomaly. 
  - Include the main cause or affected component.
  - Limit the explanation to one or two short sentences; do not write long paragraphs.

Output format (strictly follow this, no explanations beyond what is requested):
1. normal
2. abnormal - brief explanation
3. normal
4. abnormal - brief explanation
...

Logs:
{indexed_logs}

Now produce the classifications including a brief explanation for abnormal lines.
""".strip()

    messages = [
        {
            "role": "system",
            "content": "You are Qwen, a helpful assistant specialized in system log anomaly detection."
        },
        {
            "role": "user",
            "content": prompt_text
        }
    ]
    return messages

In [6]:
def parse_model_output(model_output):
    labels = []
    lines = model_output.strip().split("\n")
    
    for line in lines:
        line_lower = line.lower().strip()

        if "abnormal" in line_lower:
            labels.append(1)
        elif "normal" in line_lower:
            labels.append(0)
        else:
            labels.append(1)
    
    return labels


In [7]:
def read_ground_truth(dataset_name, dataset_dict):
    path = dataset_dict[dataset_name]
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    
    labels = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line == "":
                continue
            # anomaly=1, normal=0
            labels.append(int(line))
    return labels

In [8]:
def evaluate_predictions(model_output, dataset_name, dataset_dict):
    y_pred = parse_model_output(model_output)
    y_true = read_ground_truth(dataset_name, dataset_dict)

    if len(y_pred) != len(y_true):
        print(f"Warning: prediction length {len(y_pred)} != ground truth length {len(y_true)}")
        min_len = min(len(y_pred), len(y_true))
        y_pred = y_pred[:min_len]
        y_true = y_true[:min_len]

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)

    print(f"=== Evaluation {dataset_name} Results ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")

### Android

In [9]:
selected_dataset = "android"
log_lines = read_log_file(selected_dataset, log_datasets)

messages = build_messages(log_lines)

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [10]:
print(response)

(1) normal
(2) abnormal - Uncaught JavaScript error indicates a potential application bug.
(3) normal
(4) abnormal - Failed to get a socket from the table suggests network or resource issue.
(5) abnormal - Repeated error in getting temp info, possible issue with file access.
(6) abnormal - Repeated error in getting temp info, possible issue with file access.
(7) normal
(8) abnormal - Repeated error in getting temp info, possible issue with file access.
(9) normal
(10) abnormal - Failed to open cmdline file, could indicate file system or process issue.
(11) normal
(12) abnormal - Repeated error in getting temp info, possible issue with file access.
(13) normal
(14) abnormal - Repeated error in getting temp info, possible issue with file access.
(15) abnormal - Repeated error in getting temp info, possible issue with file access.
(16) normal
(17) abnormal - Repeated error in getting temp info, possible issue with file access.
(18) normal
(19) abnormal - Failure to check perfhub service i

In [11]:
evaluate_predictions(response, selected_dataset, labels_datasets)

=== Evaluation android Results ===
Accuracy:  0.8400
Precision: 0.8000
Recall:    0.9231
F1-score:  0.8571


### Apache

In [12]:
selected_dataset = "apache"
log_lines = read_log_file(selected_dataset, log_datasets)

messages = build_messages(log_lines)

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [13]:
print(response)

(1) abnormal - Error message indicating file does not exist.
(2) abnormal - mod_jk child workerEnv in error state.
(3) normal
(4) normal
(5) normal
(6) abnormal - Error message indicating script not found or unable to stat.
(7) abnormal - Error message indicating file does not exist.
(8) abnormal - Error message indicating file does not exist.
(9) normal
(10) abnormal - mod_jk child workerEnv in error state.
(11) abnormal - mod_jk child workerEnv in error state.
(12) normal
(13) abnormal - mod_jk child workerEnv in error state.
(14) normal
(15) abnormal - Error message indicating script not found or unable to stat.
(16) normal
(17) abnormal - Error message indicating file does not exist.
(18) abnormal - mod_jk child workerEnv in error state.
(19) abnormal - Error message indicating directory index forbidden by rule.
(20) normal
(21) abnormal - Error message indicating script not found or unable to stat.
(22) abnormal - Error message indicating directory index forbidden by rule.
(23) no

In [14]:
evaluate_predictions(response, selected_dataset, labels_datasets)

=== Evaluation apache Results ===
Accuracy:  0.9200
Precision: 0.8667
Recall:    1.0000
F1-score:  0.9286


### Proxifier

In [27]:
selected_dataset = "proxifier"
log_lines = read_log_file(selected_dataset, log_datasets)

messages = build_messages(log_lines)

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [28]:
print(response)

(1) normal
(2) abnormal - Could not connect to proxy, error 10061.
(3) abnormal - Could not connect to proxy, error 10061.
(4) normal
(5) abnormal - Could not connect to proxy, error 10061.
(6) abnormal - Could not connect to proxy, status code 503.
(7) abnormal - Could not connect to proxy, error 10061.
(8) abnormal - Connection request was canceled.
(9) normal
(10) normal
(11) abnormal - Connection request was canceled.
(12) normal
(13) abnormal - Could not resolve proxy server, error 11001.
(14) abnormal - Could not connect to proxy, error 10061.
(15) normal
(16) abnormal - Could not resolve proxy server, error 11001.
(17) abnormal - Could not connect to proxy, status code 403.
(18) normal
(19) abnormal - Could not connect to proxy, error 10061.
(20) abnormal - Connection lifetime less than 1 second.
(21) normal
(22) abnormal - Could not connect to proxy, error 10061.
(23) abnormal - Connection lifetime less than 1 second.
(24) normal
(25) abnormal - Connection request was canceled.

In [29]:
evaluate_predictions(response, selected_dataset, labels_datasets)

=== Evaluation proxifier Results ===
Accuracy:  0.8800
Precision: 0.8125
Recall:    1.0000
F1-score:  0.8966
